# Choose Your Own Basin

This tutorial provides an example of how to integrate RAPID2 into a workflow using `Python` to handle model setup and examination of model output. The code below allows for the selection of several model parameters at the start, such as basin, simulation date, and input runoff model, and then performs model setup automatically. Several python packages are used in this tutorial, but all are included in the RAPID2 pypi package, including `toyplot`, a lightweight plotting package.

This tutorial is provided as an interactive python notebook (`ipynb`), and can be opend and run in your browser by clicking the Binder badge:

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/c-h-david/rapid-hub/main?labpath=docs/user-guide/examples/tutorial-anyBasin.ipynb)

Before starting, esnure that RAPID2 has been installed in your virtual environment (see the [Quickstart Guide](https://c-h-david.github.io/rapid-hub/user-guide/quick-start-sandbox/)). If running this tutorial in Binder, this setup is done automatically.

First, import Python packages needed for the tutorial.

In [ ]:
import zipfile
from pathlib import Path
import urllib.request
import netCDF4
import numpy as np
import toyplot

import rapid2 as r2

We can check the version of the installed RAPID2 version using:

In [ ]:
print(r2.__version__)

## 1. Set Parameters
Here is where we set the parameters for the run:
- *BASIN*: Set the basin ID based on global Pfafstetter Level 2 hydrologic regions (e.g. pfaf_74 for the Mississippi Basin)
- *PHASE*: Phase of GLDAS forcing to use (e.g. 2.1)
- *MODEL*: The hydrologic model to use for input (options are: VIC, CLSM, NOAH)
- *TIME*: The year and month for the simulation (e.g.'2010-01')

Note that all input paramaters must be provided as strings.

In [ ]:
BASIN = "pfaf_71"
PHASE = '2.1'
MODEL = 'VIC'
TIME = '2010-01'

Next we set up the working directory, and input and output directories.

In [ ]:
WORK_DIR = Path("rapid2_tutorial")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Donwload Static River Parameters

Next, we need to get six static parameter files for the Mississippi Basin. We'll be using prepared files from a Zenodo repository:

[RAPID2 static files for global implementation with MERIT Basins v1.0 and GLDAS v2.0](https://zenodo.org/records/20672740)

This repository includes 61 zipped folders corresponding to the global Pfafstetter Level 2 hydrologic regions. The code below uses the pfaf ID set in the `BASIN` variable above to download and unpack the corresponding zipped folder.

In [ ]:
ZENODO_RECORD = "20672740"
ZENODO_BASE = f"https://zenodo.org/records/{ZENODO_RECORD}/files"

print(f"Downloading {BASIN} ZIP file from Zenodo record {ZENODO_RECORD}...")
url = f"{ZENODO_BASE}/MERIT_Basins_v1.0_{BASIN}_GLDAS_v2.0_RAPID_v2.0.zip?download=1"
dest = INPUT_DIR / BASIN 
if not dest.exists():
    urllib.request.urlretrieve(url, dest)
else:
    print(f'\tFiles already downloaded')
with zipfile.ZipFile(dest, "r") as z:
        z.extractall(INPUT_DIR)
print(f"\nStatic files for {BASIN}:")
for f in sorted(INPUT_DIR.glob(f"*{BASIN}*")):
    print(f"  {f.name}")

## 3. Downlod NLDAS Runoff Data

The RAPID2 routing model requires runoff inputs from a land surface model. We'll use runoff data downloaded from the Global Land Data Assimilation System (GLDAS) using the `pull.gldas` function that is part of the `rapid2` package.

A NASA Earthdata account is required to download the GLDAS data. If you have not yet done so, you may also need to accept an end-user license agreement from the NASA Goddard Earth Sciences (GES) Data and Information Services Center (DISC). The `pull.gldas` function handles Earthdata authentication with user input username and password.

In [ ]:
GLDAS_FILE = INPUT_DIR / f'GLDAS_{PHASE}_{MODEL}_{TIME}.nc4'
r2.pull.gldas2(
    YS_phs=PHASE, 
    YS_mod=MODEL, 
    YS_tim=TIME, 
    lsm_ncf=GLDAS_FILE
    )

The new file is placed in `input/Tutorial`. Other files are also automatically
downloaded in that directory and removed after use.

## 4. Couple the Runoff Data with the River Network 

Once you have the raw land surface model data and the static river basin parameters, the two must be linked by mapping the runoff to the river network.. The `prep.couple` function transforms the gridded runoff data into RAPID2's reach-based external inflow format. By providing the land surface model data, connectivity, coordinates, and coupling files, this tool generates the external inflow NetCDF file required for routing.

In [ ]:
QEXT_FILE = INPUT_DIR / f"Qext_{BASIN}_{PHASE}_{MODEL}_{TIME}.nc4"
r2.prep.couple(
    lsm_ncf=GLDAS_FILE, 
    con_pqt=f'{INPUT_DIR}/con_{BASIN}.parquet', 
    crd_pqt=f'{INPUT_DIR}/crd_{BASIN}.parquet', 
    cpl_pqt=f'{INPUT_DIR}/cpl_{BASIN}_GLDAS.parquet', 
    Qex_ncf=QEXT_FILE
    )

## 5. Create Cold Start Initial Discharge File

At the start of our simulation, we need to provide RAPID2 with initial values of discharge. The `prep.coldinit` function creates a cold start file with all initial discharge set to zero.

In [ ]:
QINIT_FILE = INPUT_DIR / f"Qinit_{BASIN}_{PHASE}_{MODEL}_{TIME}.nc4"
r2.prep.coldinit(
    Qex_ncf=QEXT_FILE,
    Q00_ncf=QINIT_FILE
    )

The resulting QINIT file is created in the input directory.

## 6. Run RAPID2 Routing Model

Finally, the RAPID2 model can be run using the `core.run` function. The output discharge files are written to the output directory.

In [ ]:
r2.core.run(
    Qex_ncf=f'./{QEXT_FILE}',
    Q00_ncf=f'./{QINIT_FILE}',
    con_pqt=f'./{INPUT_DIR}/con_{BASIN}.parquet',
    kpr_pqt=f'./{INPUT_DIR}/kpr_{BASIN}_nrm.parquet',
    xpr_pqt=f'./{INPUT_DIR}/xpr_{BASIN}_nrm.parquet',
    bas_pqt=f'./{INPUT_DIR}/bas_{BASIN}_topo.parquet',
    IS_dtR=900,
    Qou_ncf=f'./{OUTPUT_DIR}/Qout_{BASIN}_GLDAS_{PHASE}_{MODEL}_{TIME}_tst.nc4',
    Qfi_ncf=f'./{OUTPUT_DIR}/Qfinal_{BASIN}_GLDAS_{PHASE}_{MODEL}_{TIME}_tst.nc4',
)

## 7. Plot Output
Now we can examine the output dishcarge. In this tutorial, we use the `netCDF4` python package to read our modeled output discharge file. This file contains time series of discharge for our selected month with time step set by the `IS_dtR` parameter in the namelist file. Here, we compute the mean discharge of every river reach in the basin, and then plot the discharge of reach with the highest mean.

In [ ]:
with netCDF4.Dataset(f'./{OUTPUT_DIR}/Qout_{BASIN}_GLDAS_{PHASE}_{MODEL}_{TIME}_tst.nc4','r') as ds:
    qout = ds.variables["Qout"][:]
    rivid = ds.variables["rivid"][:]
    time = ds.variables['time'][:]

    mean_per_reach = np.mean(qout, axis=0)
    main = int(np.argmax(mean_per_reach))
    peak_q=float(np.max(qout[:, main]))

    canvas = toyplot.Canvas(width=900,height=400,)
    axes = canvas.cartesian(label=f'Outflow for {BASIN} Time:{TIME}',xlabel='Date',ylabel=r'Discharge $[m^3/s]$')
    mark = axes.plot(time,qout[:,main])
    
    axes.x.ticks.show = True
    axes.x.ticks.locator = toyplot.locator.Timestamp()